# brier — M3.4 benchmark: raw vs L0 on banking20 (Qwen3-1.7B)

Runs `python -m brier.bench` on the full **banking20** task (BANKING77, CC-BY-4.0):
500 unlabelled prior-pool states and 2,603 test items, levels `raw` and `L0`, flip rate via
the reversed option order, 95 % bootstrap CIs (1,000 resamples) and paired L0 − raw comparisons.

**Runtime:** set *Runtime → Change runtime type → A100 GPU*. Expect roughly 20–40 min for the
full run (~129k forward passes). Step 3 times a 50-item run first and prints an estimate.

**Reproducibility:** brier is installed from a pinned commit and the model from a pinned
revision; both are recorded in the result JSON.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Configuration and install

In [ ]:
BRIER_COMMIT = "3d717e34d1d1bebfbe1301d4b5c673da9b4b5d4b"  # brier main after PR #9
MODEL = "Qwen/Qwen3-1.7B"
REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"  # pinned model commit
DTYPE = "bfloat16"  # A100/L4 support bf16; on a T4 use "float16"
BATCH = 64  # prompts per forward pass
OUT = "/content/results"

In [ ]:
!git clone -q https://github.com/mohdUwaish59/brier.git /content/brier
!git -C /content/brier checkout -q {BRIER_COMMIT}
!pip install -q -e "/content/brier[hf]"
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

## 3. Timing run (50 items)

Uses the first 50 test items and 50 pool states. Takes a few minutes and downloads the model
(~3.4 GB) and the dataset (~0.8 MB, SHA-256 verified).

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype {DTYPE} \
    --batch-size {BATCH} --limit 50 --n-resamples 200 --out /content/results_timing

In [ ]:
import glob
import json

with open(glob.glob("/content/results_timing/*.json")[0]) as f:
    timing = json.load(f)
w = timing["timing"]["wall_seconds"]
# Full run: raw scales with test items; L0 with pool + test items (x20 rotations x2 orders).
raw_est = w["raw"] * 2603 / 50
l0_est = w["L0"] * (500 + 2603) / (50 + 50)
print(f"timing run: raw {w['raw']:.0f}s, L0 {w['L0']:.0f}s")
print(
    f"estimated full run: raw ~{raw_est / 60:.0f} min, L0 ~{l0_est / 60:.0f} min, "
    f"total ~{(raw_est + l0_est) / 60:.0f} min"
)

## 4. Full run

Keep this tab open. Results are written after each level (raw first), so if the session drops
during L0 the raw results are already saved in `/content/results`.

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype {DTYPE} \
    --batch-size {BATCH} --out {OUT}

## 5. Results

In [ ]:
import glob
import json

with open(glob.glob(f"{OUT}/*.json")[0]) as f:
    res = json.load(f)


def fmt(v):
    return f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]"


print(f"model {res['model']['id']}@{res['model']['revision'][:8]}  brier {res['git_commit']}")
print(
    f"n_test={res['n_test']} n_pool={res['n_pool']} device={res['env']['device']} "
    f"dtype={res['model']['dtype']}"
)
print()
print("| metric | raw | L0 | L0 - raw (paired) |")
print("|---|---|---|---|")
for k in res["levels"]["raw"]:
    raw, l0 = res["levels"]["raw"][k], res["levels"]["L0"][k]
    diff = res["comparisons"]["L0_minus_raw"][k]
    print(f"| {k} | {fmt(raw)} | {fmt(l0)} | {fmt(diff)} |")
print()
print("wall seconds:", res["timing"]["wall_seconds"])

# M3.4 acceptance: L0 flip rate lower than raw with non-overlapping CIs.
raw_f, l0_f = res["levels"]["raw"]["flip_rate"], res["levels"]["L0"]["flip_rate"]
ok = l0_f[2] < raw_f[1]
print("M3.4 acceptance (L0 flip-rate CI entirely below raw):", "PASS" if ok else "FAIL")

## 6. Download the results (JSON + per-item `.npz`, no state text)

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/brier_m34_results", "zip", OUT)
files.download("/content/brier_m34_results.zip")